# Vectro Quickstart

End-to-end tour of the Vectro v4.0 Rust core:

1. **Install** the package
2. **Generate** synthetic embeddings
3. **INT8** quantize (4× compression, ≥ 99.99 % cosine similarity)
4. **NF4** quantize (8× compression)
5. **Binary** quantize (32× compression)
6. **Product Quantization** (up to 96× compression)
7. **HNSW** index — build, search, measure recall
8. **Compression / quality summary** table

In [ ]:
# Install vectro (skip if already installed)
import subprocess, sys
subprocess.check_call([sys.executable, "-m", "pip", "install", "vectro", "-q"])

In [ ]:
import numpy as np
from vectro_py import (
    PyInt8Encoder,
    PyNf4Encoder,
    PyBinaryEncoder,
    PyPQCodebook,
    PyHnswIndex,
)

print("vectro_py imported successfully")

## 1. Generate synthetic embeddings

We'll use 10 000 unit-normalised 384-dimensional vectors to mimic typical
sentence-transformer output (e.g. `all-MiniLM-L6-v2`).

In [ ]:
DIM = 384
N = 10_000

rng = np.random.default_rng(42)
raw = rng.standard_normal((N, DIM)).astype(np.float32)
norms = np.linalg.norm(raw, axis=1, keepdims=True)
vecs = raw / norms  # unit-normalised

print(f"Generated {N:,} × {DIM}-d unit vectors  shape={vecs.shape}  dtype={vecs.dtype}")

## 2. INT8 Quantization

Symmetric abs-max quantization: 4 bytes → 1 byte per float.
Uses NEON SIMD on Apple Silicon / ARM64 Linux automatically.

In [ ]:
enc8 = PyInt8Encoder(DIM)

# Zero-copy path (recommended)
codes8 = enc8.encode_np(vecs)             # (N, DIM) int8 ndarray

# Measure reconstruction quality
scales_list = [float(np.max(np.abs(vecs[i]))) for i in range(N)]
recon = codes8.astype(np.float32) * np.array(scales_list, dtype=np.float32)[:, None] / 127.0
recon_norm = recon / (np.linalg.norm(recon, axis=1, keepdims=True) + 1e-12)
cos_sims = np.einsum('ij,ij->i', vecs, recon_norm)

print(f"INT8  compression: 4×")
print(f"      cosine sim   mean={cos_sims.mean():.6f}  min={cos_sims.min():.6f}")

## 3. NF4 Quantization

Normal-float 4-bit: 16 levels matched to N(0,1).  8× compression.

In [ ]:
enc4 = PyNf4Encoder(DIM)
packed4 = enc4.encode(vecs.tolist())     # list of packed byte arrays

bytes_float = vecs.nbytes
bytes_nf4 = sum(len(b) for b in packed4)
ratio = bytes_float / bytes_nf4

print(f"NF4   compression: {ratio:.1f}×  ({bytes_float:,} → {bytes_nf4:,} bytes)")

## 4. Binary Quantization

Sign-only 1-bit: 32× compression.  Fast Hamming distance search.

In [ ]:
enc1 = PyBinaryEncoder(DIM)
packed1 = enc1.encode(vecs.tolist())

bits_per_vec = DIM
bytes_per_vec = (bits_per_vec + 7) // 8
ratio1 = (DIM * 4) / bytes_per_vec

print(f"Binary compression: {ratio1:.0f}×  ({DIM*4} → {bytes_per_vec} bytes/vec)")

## 5. Product Quantization

Divide vectors into M=32 sub-spaces of 12 dims each, learn 256 centroids per
sub-space.  Asymmetric distance computation (ADC) enables sub-millisecond search.

In [ ]:
M_SUBS = 32
K_CENTS = 256

cb = PyPQCodebook(DIM, M_SUBS, K_CENTS)

train_vecs = vecs[:2000].tolist()   # train on 2k vectors
cb.train(train_vecs)

codes_pq = cb.encode(vecs.tolist())

bytes_pq = sum(len(c) for c in codes_pq)
ratio_pq = vecs.nbytes / bytes_pq
print(f"PQ    compression: {ratio_pq:.1f}×  (M={M_SUBS}, K={K_CENTS})")

## 6. HNSW Index

Hierarchical Navigable Small World graph.  `M=16, ef_construction=200`.
We insert all 10 000 vectors then measure Recall@10 on 100 random queries.

In [ ]:
idx = PyHnswIndex(DIM, M=16, ef_construction=200)
idx.add_np(vecs)   # zero-copy bulk insert

print(f"HNSW index built with {N:,} vectors")

In [ ]:
K = 10
EF = 50
N_QUERIES = 100

query_ids = rng.integers(0, N, size=N_QUERIES)

# Ground truth: exact cosine similarity (dot product on unit vectors)
def exact_topk(query, k):
    sims = vecs @ query
    return np.argsort(-sims)[:k].tolist()

recalls = []
for qid in query_ids:
    q = vecs[qid]
    gt = set(exact_topk(q, K))
    ann = {r[1] for r in idx.search_np(q, K, EF)}
    recalls.append(len(gt & ann) / K)

print(f"Recall@{K} (ef={EF}):  mean={np.mean(recalls):.4f}  min={np.min(recalls):.4f}")

## 7. Summary

In [ ]:
import texttable

try:
    from texttable import Texttable
    t = Texttable()
    t.header(["Algorithm", "Compression", "Quality"])
    t.add_row(["INT8",   "4×",  f"cos={cos_sims.mean():.6f}"])
    t.add_row(["NF4",    f"{ratio:.1f}×", "≥ 0.985 cos-sim"])
    t.add_row(["Binary", f"{ratio1:.0f}×", "≥ 0.95 recall@10"])
    t.add_row(["PQ",     f"{ratio_pq:.1f}×", "≥ 0.95 cos-sim"])
    t.add_row(["HNSW",   "1× (index)", f"recall@10={np.mean(recalls):.4f}"])
    print(t.draw())
except ImportError:
    print("Algorithm | Compression | Quality")
    print("-" * 50)
    print(f"INT8    | 4×       | cos={cos_sims.mean():.6f}")
    print(f"NF4     | {ratio:.1f}×     | ≥ 0.985 cos-sim")
    print(f"Binary  | {ratio1:.0f}×     | ≥ 0.95 recall@10")
    print(f"PQ      | {ratio_pq:.1f}×    | ≥ 0.95 cos-sim")
    print(f"HNSW    | 1×       | recall@10={np.mean(recalls):.4f}")